<a href="https://colab.research.google.com/github/SalesforceAIResearch/SlackAgents/blob/main/docs/docs/examples/tools/tool_definition_from_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Tools from LangChain Toolkit ⛓️‍💥
This notebook demonstrates how to use tools from the LangChain toolkit in SlackAgents. We'll cover:
- Defining a custom tool in LangChain
- Using the custom LangChain tool in SlackAgents
- Using LangChain tools from the LangChain community

The LangChain community has developed an extensive number of AI agent tools. LangChain's [Tookit](https://python.langchain.com/v0.1/docs/integrations/toolkits/) integrates tools from open-source communities. As a result, an alternative way to jumpstart agent development is by using these ready-made tools. SlackAgents supports this by directly enabling usage of tools from external libraries.

`SlackAgents` support the use of tools from the `LangChain` toolkit. In this notebook, we will show how to define a tool in LangChain and use it in `SlackAgent`.

If you're opening this Notebook on colab, you will probably need to install `SlackAgents` and `LangChain`. You can do this by running the following cell:

```python
!pip install slackagents
!pip install langchain-core lanchain-openai langchain-communities
```

In [ ]:
%pip install slackagents
%pip install slackagents langchain langchain-community wikipedia

## Define a Custom Tool in LangChain


Let's create a simple multiplication tool using LangChain's `BaseTool` class:


In [1]:
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

class MultiplyInput(BaseModel):
    a: int = Field(description="first number")
    b: int = Field(description="second number")
    
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "useful for when you need to calculate the product of two numbers"
    args_schema: Type[BaseModel] = MultiplyInput
    return_direct: bool = True

    def _run(
        self, a: int, b: int
    ) -> str:
        """Use the tool."""
        return a * b
langchain_multiply_tool = MultiplyTool()

## Use the Custom LangChain Tool in `SlackAgents`
Now, let's wrap the LangChain tool for use in SlackAgents:

In [2]:

from slackagents.tools.wrappers.langchain import LangChainToolWrapper
from slackagents.tools.base import ToolCall
import json

multiply_tool = LangChainToolWrapper(langchain_multiply_tool)
print(json.dumps(multiply_tool.info, indent=4))
tool_call = ToolCall(name="multiply", arguments={"a": 2, "b": 3})
print(f"The result of the multiplication is: {multiply_tool.execute(tool_call)}")


{
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "useful for when you need to calculate the product of two numbers",
        "parameters": {
            "properties": {
                "a": {
                    "description": "first number",
                    "type": "integer"
                },
                "b": {
                    "description": "second number",
                    "type": "integer"
                }
            },
            "required": [
                "a",
                "b"
            ],
            "type": "object"
        }
    }
}
The result of the multiplication is: 6


## Use a Pre-built LangChain Tool (Wikipedia) in `SlackAgents`
Let's use the Wikipedia tool from LangChain:

In [3]:
# Create the Wikipedia tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
langchain_wikipedia_tool = WikipediaQueryRun(api_wrapper=api_wrapper)
# Wrap the tool for use in SlackAgents
wikipedia_tool = LangChainToolWrapper(langchain_wikipedia_tool)
# Execute the tool
tool_call = ToolCall(name="wikipedia", arguments={"query": "Salesforce"})

In [6]:
print(json.dumps(wikipedia_tool.info, indent=4))

{
    "type": "function",
    "function": {
        "name": "wikipedia",
        "description": "A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.",
        "parameters": {
            "properties": {
                "query": {
                    "description": "query to look up on wikipedia",
                    "type": "string"
                }
            },
            "required": [
                "query"
            ],
            "type": "object"
        }
    }
}


In [7]:
result = wikipedia_tool.execute(tool_call)
print(f"Wikipedia result: {result}")

Wikipedia result: Page: Salesforce
Summary: Salesforce, Inc. is an American cloud-based software company headquartered


This notebook demonstrates how to integrate LangChain tools with SlackAgents, allowing you to leverage the extensive toolkit provided by LangChain in your SlackAgents applications.